# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# Para árboles: solo imputación por mediana (no necesitan escalar)
tree_preprocessor = ColumnTransformer(
    [("num", SimpleImputer(strategy="median"), feature_columns)]
)

# Para modelos sensibles a la escala (regresión logística, KNN): imputación + estandarización
scaled_preprocessor = ColumnTransformer(
    [("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), feature_columns)]
)

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Porque si se ajusta el `StandardScaler` con todo el entrenamiento **antes** de la validación cruzada, la media y la desviación estándar ya incorporan información de los datos de validación de cada fold: hay **fuga de información (data leakage)** y las métricas salen optimistas, porque el fold de validación deja de ser realmente «no visto». Dentro del `Pipeline`, en cada fold el escalador se ajusta **solo con los datos de entrenamiento de ese fold** y luego se aplica al de validación, imitando lo que pasará con datos nuevos. Además, el pipeline guarda el preprocesamiento junto al modelo, así que en inferencia se aplican exactamente las mismas transformaciones.

## 3. Modelos candidatos

In [5]:
models = {
    "decision_tree": Pipeline([
        ("prep", clone(tree_preprocessor)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline([
        ("prep", clone(scaled_preprocessor)),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    "knn": Pipeline([
        ("prep", clone(scaled_preprocessor)),
        ("model", KNeighborsClassifier()),
    ]),
}
list(models)

['decision_tree', 'logistic_regression', 'knn']

## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipe in models.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring="accuracy", return_train_score=True)
    rows.append({
        "model": name,
        "train_acc": res["train_score"].mean(),
        "val_acc": res["test_score"].mean(),
        "val_std": res["test_score"].std(),
    })

initial_comparison = pd.DataFrame(rows).set_index("model").round(4)
initial_comparison

,train_acc,val_acc,val_std
model,,,
decision_tree,1.0000,0.9022,0.0586
logistic_regression,1.0000,0.9791,0.0277
knn,0.9771,0.9507,0.0353


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "decision_tree": {
        "model__max_depth": [2, 3, 4, 5, None],
        "model__min_samples_leaf": [1, 3, 5],
    },
    "logistic_regression": {
        "model__C": [0.01, 0.1, 1, 10, 100],
    },
    "knn": {
        "model__n_neighbors": [3, 5, 7, 9, 11],
        "model__weights": ["uniform", "distance"],
    },
}

searches = {}
rows = []
for name, pipe in models.items():
    gs = GridSearchCV(pipe, param_grids[name], scoring="accuracy", cv=cv,
                      return_train_score=True, refit=True)
    gs.fit(X_train, y_train)
    searches[name] = gs
    i = gs.best_index_
    rows.append({
        "model": name,
        "best_cv_acc": gs.best_score_,
        "cv_std": gs.cv_results_["std_test_score"][i],
        "train_acc": gs.cv_results_["mean_train_score"][i],
        "best_params": gs.best_params_,
    })

tuned_comparison = pd.DataFrame(rows).set_index("model")
tuned_comparison[["best_cv_acc", "cv_std", "train_acc"]] = tuned_comparison[["best_cv_acc", "cv_std", "train_acc"]].round(4)
tuned_comparison

,best_cv_acc,cv_std,train_acc,best_params
model,,,,
decision_tree,0.9022,0.0586,0.9982,"{'model__max_depth': 4, 'model__min_samples_le..."
logistic_regression,0.9791,0.0277,1.0000,{'model__C': 1}
knn,0.9579,0.0405,0.9648,"{'model__n_neighbors': 7, 'model__weights': 'u..."


## 6. Selección y persistencia del campeón

In [8]:
import sklearn

# Mayor best_score_ de CV; si hay empate, se prefiere el de menor desviación estándar
ranking = tuned_comparison.sort_values(["best_cv_acc", "cv_std"], ascending=[False, True])
champion_name = ranking.index[0]
champion = searches[champion_name].best_estimator_      # pipeline completo, ya reajustado con todo train
print("Campeón:", champion_name, "|", searches[champion_name].best_params_)
print(ranking[["best_cv_acc", "cv_std"]])

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

best = tuned_comparison.loc[champion_name]
metadata = {
    "champion": champion_name,
    "best_params": {k: (None if v is None else (v.item() if hasattr(v, "item") else v))
                    for k, v in searches[champion_name].best_params_.items()},
    "cv_accuracy": float(best["best_cv_acc"]),
    "cv_std": float(best["cv_std"]),
    "cv_train_accuracy": float(best["train_acc"]),
    "cv_folds": 5,
    "scoring": "accuracy",
    "random_state": RANDOM_STATE,
    "n_train": int(len(X_train)),
    "features": feature_columns,
    "sklearn_version": sklearn.__version__,
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

tuned_comparison.assign(best_params=tuned_comparison["best_params"].astype(str)).to_csv(REPORTS_DIR / "cv_results.csv")
print("\nArtefactos guardados.")

Campeón: logistic_regression | {'model__C': 1}
                     best_cv_acc  cv_std
model                                   
logistic_regression       0.9791  0.0277
knn                       0.9579  0.0405
decision_tree             0.9022  0.0586

Artefactos guardados.


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> **Campeón: regresión logística (`C = 1`, con `StandardScaler`).**
>
> - **Desempeño:** tiene la mayor accuracy en CV (**0.9791**), por encima de KNN (0.9579) y del árbol (0.9022).
> - **Variabilidad:** también es el más estable entre folds (desv. estándar **0.0277**, frente a 0.0405 de KNN y 0.0586 del árbol).
> - **Sobreajuste:** el árbol es el que más sobreajusta (train 0.998 vs validación 0.902, brecha ≈ 0.10). La regresión logística tiene una brecha pequeña (train 1.000 vs 0.979, ≈ 0.02) y KNN una brecha mínima (0.965 vs 0.958), pero con peor desempeño.
> - **Complejidad e interpretabilidad:** la regresión logística es un modelo lineal simple, con un solo hiperparámetro, rápido, que entrega probabilidades y cuyos coeficientes se pueden interpretar. KNN debe guardar todos los datos y depende de la escala; el árbol es interpretable pero inestable (alta varianza).
>
> **Salvedad:** con solo 142 muestras de entrenamiento, la diferencia entre la regresión logística y KNN (≈ 2 puntos) es del orden de la desviación estándar entre folds, por lo que no es una ventaja contundente. Aun así, la regresión logística gana a la vez en desempeño, estabilidad y simplicidad, y el criterio de selección (mayor `best_score_`, con menor desviación como desempate) se fijó antes de abrir el test.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.